# Flow Matching on 32?32 Mazes

Compare how much random weight initialization affects two flow-matching objectives. Train **10 independently initialized models per objective**?velocity MSE and endpoint-residual MSE?and evaluate every model with the same Gaussian starts and 20 Euler ODE steps.

Each replicate pair shares its initial weights and training examples, so the comparison can show both initialization variability and differences between objectives. Solvability is measured as the fraction of generated mazes whose entrance connects to the exit.


In [18]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import copy
import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display, Markdown
import ipywidgets as widgets
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import (
    Denoiser,
    decode_probabilities,
    embed_binary,
    make_binary_embeddings,
)

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [19]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

## 2. Forward Flow Path

For a clean embedded maze `x_data` and independent Gaussian noise `z`, define the conditional path

`x_t = (1 - t) * z + t * x_data`, with `t` sampled uniformly from `[0, 1]`.

Its target velocity is constant along the path: `v_target = x_data - z`. This panel decodes the continuous states using the same fixed-temperature embedding dot-product visualization at every time; the visualization does not alter the path.

In [20]:
EMBEDDING_DIM = 2
TEMPERATURE = 0.5
binary_embeddings = make_binary_embeddings(EMBEDDING_DIM, DEVICE)

preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
x_data = embed_binary(preview_maze, binary_embeddings)
preview_noise = torch.randn_like(x_data)
preview_times = [0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
with torch.no_grad():
    preview_frames = []
    for t in preview_times:
        t_batch = torch.full((1,), t, device=DEVICE)
        x_t = (1.0 - t_batch[:, None, None, None]) * preview_noise + t_batch[
            :, None, None, None
        ] * x_data
        probabilities = decode_probabilities(x_t, binary_embeddings, TEMPERATURE)
        preview_frames.append(probabilities[0, 1].cpu().numpy())

path_fig = make_subplots(
    rows=1,
    cols=len(preview_times),
    subplot_titles=[f"t={t:g}" for t in preview_times],
)
for col, frame in enumerate(preview_frames, start=1):
    path_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=col,
    )
path_fig.update_layout(
    title="Linear flow path: black = p(0), white = p(1)",
    width=1700,
    height=360,
)
path_fig.update_xaxes(showticklabels=False)
path_fig.update_yaxes(showticklabels=False, autorange="reversed")
path_fig.show()

## 3. Model Architecture

The U-Net structure is shared with the DDPM notebook: input shape `(B, 2, 32, 32)`, base channels `64` with encoder widths `64 -> 128 -> 256`, residual blocks conditioned by a `128`-dimensional time embedding, and a two-channel output. Here the output represents velocity `v(x_t, t)`, not Gaussian noise. The normalized flow time is scaled to `[0, 100]` before sinusoidal embedding so the existing time encoder receives a useful range. See `notebooks/_shared/model.py` for the implementation.

In [21]:
BASE_CHANNELS = 64
TIME_DIM = 128
TIME_EMBEDDING_SCALE = 100.0
N_REPLICATES = 10
OBJECTIVES = ("velocity", "endpoint_residual")
OBJECTIVE_LABELS = {
    "velocity": "Velocity MSE",
    "endpoint_residual": "Endpoint-residual MSE",
}
OBJECTIVE_COLORS = {
    "velocity": "#4263eb",
    "endpoint_residual": "#e76f00",
}

# Pair the two objectives by initialization seed; each pair starts identically.
models = {}
MODEL_META = {}
for replicate in range(N_REPLICATES):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED + 1000 + replicate)
        initial_model = Denoiser(
            embedding_dim=EMBEDDING_DIM,
            base_channels=BASE_CHANNELS,
            time_dim=TIME_DIM,
        )
    for objective in OBJECTIVES:
        key = f"{objective}_{replicate + 1:02d}"
        candidate = (
            initial_model if objective == "velocity" else copy.deepcopy(initial_model)
        )
        models[key] = candidate.to(DEVICE)
        MODEL_META[key] = {"objective": objective, "replicate": replicate + 1}

## 4. Compare two flow-matching objectives across initializations

Train 10 paired replicates for each objective. Within each pair, both models start from identical weights and see the same maze, noise, and time batches.

1. **Velocity MSE:** $\operatorname{MSE}(v_\theta, x_1-z)$.
2. **Endpoint-residual MSE:** $\operatorname{MSE}((1-t_0)v_\theta, x_1-x_{t_0})$.

Both models predict the path velocity. The second loss weights velocity errors by $(1-t_0)^2$. The training plot reports mean train and validation losses across the 10 replicates for each objective.


In [22]:
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 1

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizers = {
    name: torch.optim.AdamW(candidate.parameters(), lr=LEARNING_RATE)
    for name, candidate in models.items()
}


def sample_training_context(maze_batch, generator=None):
    x_data = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    noise = torch.randn(
        x_data.shape, generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t0 = (1.0 - 1e-3) * torch.rand(
        x_data.shape[0], generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t0_image = t0[:, None, None, None]
    x_t0 = (1.0 - t0_image) * noise + t0_image * x_data
    return x_data, noise, t0, x_t0


def objective_loss(context, candidate, objective):
    x_data, noise, t0, x_t0 = context
    predicted_velocity = candidate(x_t0, t0 * TIME_EMBEDDING_SCALE)
    if objective == "velocity":
        return F.mse_loss(predicted_velocity, x_data - noise)
    endpoint_residual = x_data - x_t0
    scale = (1.0 - t0)[:, None, None, None]
    return F.mse_loss(scale * predicted_velocity, endpoint_residual)


@torch.no_grad()
def validation_losses():
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    was_training = {name: candidate.training for name, candidate in models.items()}
    for candidate in models.values():
        candidate.eval()
    totals = {name: [] for name in models}
    try:
        for (maze_batch,) in val_loader:
            context = sample_training_context(maze_batch, generator)
            for name, candidate in models.items():
                objective = MODEL_META[name]["objective"]
                totals[name].append(
                    objective_loss(context, candidate, objective).item()
                )
        return {name: float(np.mean(values)) for name, values in totals.items()}
    finally:
        for name, candidate in models.items():
            candidate.train(was_training[name])


train_history = {objective: [] for objective in OBJECTIVES}
val_history = {objective: [] for objective in OBJECTIVES}
loss_fig = go.Figure()
for objective in OBJECTIVES:
    color = OBJECTIVE_COLORS[objective]
    loss_fig.add_scatter(
        name=f"{OBJECTIVE_LABELS[objective]} train",
        mode="lines",
        line=dict(color=color, width=2),
    )
    loss_fig.add_scatter(
        name=f"{OBJECTIVE_LABELS[objective]} validation",
        mode="lines",
        line=dict(color=color, width=2, dash="dot"),
    )
loss_fig.update_layout(
    title="Mean loss across 10 paired initializations",
    xaxis_title="Epoch",
    yaxis_title="Objective MSE (log scale; scales differ)",
    yaxis_type="log",
    template="plotly_white",
    width=1050,
    height=520,
    legend=dict(orientation="h", y=-0.22),
)

epoch_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=EPOCHS * len(train_loader) * len(models),
    description="Training",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(epoch_progress)
display(loss_fig)
for epoch in range(1, EPOCHS + 1):
    for candidate in models.values():
        candidate.train()
    epoch_values = {name: [] for name in models}
    for batch_index, (maze_batch,) in enumerate(train_loader, start=1):
        context = sample_training_context(maze_batch)
        for model_index, (name, candidate) in enumerate(models.items(), start=1):
            objective = MODEL_META[name]["objective"]
            loss = objective_loss(context, candidate, objective)
            optimizers[name].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(candidate.parameters(), max_norm=1.0)
            optimizers[name].step()
            epoch_values[name].append(loss.item())
            epoch_progress.value = (
                (epoch - 1) * len(train_loader) * len(models)
                + (batch_index - 1) * len(models)
                + model_index
            )

    for objective in OBJECTIVES:
        run_means = [
            float(np.mean(epoch_values[name]))
            for name in models
            if MODEL_META[name]["objective"] == objective
        ]
        train_history[objective].append(float(np.mean(run_means)))
    validation_by_model = validation_losses()
    for objective in OBJECTIVES:
        run_values = [
            validation_by_model[name]
            for name in models
            if MODEL_META[name]["objective"] == objective
        ]
        val_history[objective].append(float(np.mean(run_values)))

    if epoch % PLOT_EVERY == 0 or epoch == 1:
        for objective_index, objective in enumerate(OBJECTIVES):
            train_trace = 2 * objective_index
            val_trace = train_trace + 1
            loss_fig.data[train_trace].x = list(range(1, epoch + 1))
            loss_fig.data[train_trace].y = np.maximum(train_history[objective], 1e-12)
            loss_fig.data[val_trace].x = list(range(1, epoch + 1))
            loss_fig.data[val_trace].y = np.maximum(val_history[objective], 1e-12)
        loss_fig.update_layout(
            title=f"Mean loss across 10 paired initializations | epoch {epoch}/{EPOCHS}"
        )
        clear_output(wait=True)
        display(epoch_progress)
        display(loss_fig)

IntProgress(value=50400, bar_style='info', description='Training', layout=Layout(width='100%'), max=50400, sty…

## 5. Save the 20 trained models

The checkpoint stores every replicate, its objective and replicate number, the shared pixel embeddings, and the training configuration.


In [23]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_flow_matching_replicates.pt"
torch.save(
    {
        "models_state_dict": {
            name: model.state_dict() for name, model in models.items()
        },
        "model_meta": MODEL_META,
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
            "n_replicates_per_objective": N_REPLICATES,
            "objectives": list(OBJECTIVES),
            "epochs": EPOCHS,
        },
    },
    checkpoint_path,
)

## Load the 20 trained models

Run this cell after section 4 if you want to skip training and continue with the solvability comparison. It reconstructs the same paired initialization set before loading weights.


In [24]:
checkpoint_path = PROJECT_ROOT / "artifacts" / "maze_flow_matching_replicates.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
for name, candidate in models.items():
    candidate.load_state_dict(checkpoint["models_state_dict"][name])
    candidate.eval()
MODEL_META = checkpoint["model_meta"]
binary_embeddings = checkpoint["binary_embeddings"].to(DEVICE)

## 6. Solvability after 20-step ODE sampling

Evaluate every trained model on the same Gaussian initial states with exactly 20 explicit Euler steps. Each model's solvability percentage estimates its expected success rate over generated mazes.

The scatter plot shows the 20 per-model estimates. The box plot compares the distributions across initializations; a dark horizontal mark and label show each objective's mean.


In [29]:
N_METRIC_SAMPLES = 128
ODE_STEPS = 20


@torch.no_grad()
def sample_flow(candidate, initial_state, steps=ODE_STEPS):
    candidate.eval()
    state = initial_state.detach().clone()
    dt = 1.0 / steps
    for step in range(steps):
        t = step / steps
        times = torch.full((state.shape[0],), t * TIME_EMBEDDING_SCALE, device=DEVICE)
        state = state + dt * candidate(state, times)
    return state


evaluation_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 501)
evaluation_initial_states = torch.randn(
    N_METRIC_SAMPLES,
    EMBEDDING_DIM,
    32,
    32,
    generator=evaluation_rng,
    device=DEVICE,
    dtype=binary_embeddings.dtype,
)
evaluation_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(models),
    description="Solvability",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(evaluation_progress)
solvability_results = {}
for model_index, (name, candidate) in enumerate(models.items(), start=1):
    continuous = sample_flow(candidate, evaluation_initial_states, steps=ODE_STEPS)
    generated_mazes = (
        torch.einsum("bdhw,kd->bkhw", continuous, binary_embeddings)
        .argmax(dim=1)
        .cpu()
        .numpy()
        .astype(np.uint8)
    )
    solved = np.asarray([solvability_rate([maze]) == 1.0 for maze in generated_mazes])
    solvability_results[name] = {
        "objective": MODEL_META[name]["objective"],
        "replicate": MODEL_META[name]["replicate"],
        "solvable": solved,
        "mazes": generated_mazes,
        "rate": float(solved.mean()),
    }
    evaluation_progress.value = model_index

# Per-model Monte Carlo estimates as percentages.
scatter_fig = go.Figure()
for objective_index, objective in enumerate(OBJECTIVES):
    entries = [
        (name, result)
        for name, result in solvability_results.items()
        if result["objective"] == objective
    ]
    jitter = np.linspace(-0.10, 0.10, len(entries))
    scatter_fig.add_scatter(
        x=objective_index + jitter,
        y=[100 * result["rate"] for _, result in entries],
        mode="markers",
        name=OBJECTIVE_LABELS[objective],
        marker=dict(
            size=11,
            color=OBJECTIVE_COLORS[objective],
            opacity=0.8,
            line=dict(color="white", width=1),
        ),
        customdata=[[name, result["replicate"]] for name, result in entries],
        hovertemplate="%{customdata[0]} (replicate %{customdata[1]})<br>solvability=%{y:.1f}%<extra></extra>",
    )
    objective_rates = [100 * result["rate"] for _, result in entries]
    scatter_fig.add_scatter(
        x=[objective_index],
        y=[np.mean(objective_rates)],
        mode="markers",
        name=f"{OBJECTIVE_LABELS[objective]} mean",
        marker=dict(
            size=17,
            symbol="diamond",
            color=OBJECTIVE_COLORS[objective],
            line=dict(color="#111827", width=1.5),
        ),
        hovertemplate=f"Mean {OBJECTIVE_LABELS[objective]}<br>%{{y:.1f}}%<extra></extra>",
        showlegend=False,
    )
scatter_fig.update_layout(
    title=f"Per-model maze solvability after {ODE_STEPS} Euler steps ({N_METRIC_SAMPLES} shared starts)",
    xaxis=dict(
        title="Training objective",
        tickmode="array",
        tickvals=[0, 1],
        ticktext=[OBJECTIVE_LABELS[name] for name in OBJECTIVES],
        range=[-0.35, 1.35],
    ),
    yaxis=dict(title="Solvability (%)", range=[0, 100]),
    template="plotly_white",
    width=950,
    height=520,
)
scatter_fig.show()

# Model-level solvability distributions: box plots with explicit horizontal mean marks.
box_fig = go.Figure()
for objective_index, objective in enumerate(OBJECTIVES):
    entries = [
        (name, result)
        for name, result in solvability_results.items()
        if result["objective"] == objective
    ]
    values = np.asarray([100 * result["rate"] for _, result in entries])
    labels = [f"{name} ? replicate {result['replicate']}" for name, result in entries]
    color = OBJECTIVE_COLORS[objective]
    box_fig.add_trace(
        go.Box(
            x=[objective_index] * len(values),
            y=values,
            name=OBJECTIVE_LABELS[objective],
            boxpoints="all",
            jitter=0.22,
            pointpos=0,
            marker=dict(color=color, size=9, opacity=0.78),
            line=dict(color=color, width=2),
            fillcolor=f"rgba({int(color[1:3], 16)},{int(color[3:5], 16)},{int(color[5:7], 16)},0.16)",
            customdata=np.asarray(labels)[:, None],
            hovertemplate="%{customdata[0]}<br>Solvability: %{y:.1f}%<extra></extra>",
            showlegend=False,
        )
    )
    mean_value = float(values.mean())
    box_fig.add_shape(
        type="line",
        x0=objective_index - 0.18,
        x1=objective_index + 0.18,
        y0=mean_value,
        y1=mean_value,
        xref="x",
        yref="y",
        line=dict(color="#111827", width=4),
    )
    box_fig.add_annotation(
        x=objective_index,
        y=mean_value,
        text=f"mean {mean_value:.1f}%",
        showarrow=False,
        yshift=15,
        font=dict(size=14, color="#111827"),
        bgcolor="rgba(255,255,255,0.88)",
    )
box_fig.update_layout(
    title=f"Solvability across 10 initializations ? {ODE_STEPS}-step ODE",
    xaxis=dict(
        title="Training objective",
        tickmode="array",
        tickvals=[0, 1],
        ticktext=[OBJECTIVE_LABELS[name] for name in OBJECTIVES],
        range=[-0.5, 1.5],
    ),
    yaxis=dict(title="Solvability (%)", range=[0, 100], dtick=10),
    template="plotly_white",
    width=1000,
    height=520,
)
box_fig.show()

IntProgress(value=0, bar_style='info', description='Solvability', layout=Layout(width='100%'), max=20, style=P…

## 7. Solution length and A* search effort

For each generated maze, calculate the shortest entrance-to-exit path length with BFS and count A* expansions until the exit is reached. The goal cell is included in the expansion count. If the exit is unreachable, the shortest length is undefined; A* expansions count all cells searched before the frontier is exhausted.

The plots compare each model's mean over the generated mazes, grouped by training objective. The shortest-length mean is conditioned on solvable mazes.


In [32]:
from collections import deque
import heapq

MAZE_START = (0, 1)
MAZE_GOAL = (31, 30)
MAZE_DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))

# Notebook cells may be rerun out of order; refresh stale samples without mazes.
if "evaluation_initial_states" not in globals():
    evaluation_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 501)
    evaluation_initial_states = torch.randn(
        N_METRIC_SAMPLES,
        EMBEDDING_DIM,
        32,
        32,
        generator=evaluation_rng,
        device=DEVICE,
        dtype=binary_embeddings.dtype,
    )
for model_name, candidate in models.items():
    old_result = solvability_results.get(model_name)
    if old_result is None or "mazes" not in old_result:
        continuous = sample_flow(candidate, evaluation_initial_states, steps=ODE_STEPS)
        generated_mazes = (
            torch.einsum("bdhw,kd->bkhw", continuous, binary_embeddings)
            .argmax(dim=1)
            .cpu()
            .numpy()
            .astype(np.uint8)
        )
        solved = np.asarray(
            [solvability_rate([maze]) == 1.0 for maze in generated_mazes]
        )
        solvability_results[model_name] = {
            "objective": MODEL_META[model_name]["objective"],
            "replicate": MODEL_META[model_name]["replicate"],
            "solvable": solved,
            "mazes": generated_mazes,
            "rate": float(solved.mean()),
        }
solvability_results = {name: solvability_results[name] for name in models}


def shortest_path_length_and_astar_expansions(maze):
    passable = np.asarray(maze, dtype=bool)
    height, width = passable.shape

    def neighbors(cell):
        row, col = cell
        for dr, dc in MAZE_DIRECTIONS:
            nr, nc = row + dr, col + dc
            if 0 <= nr < height and 0 <= nc < width and passable[nr, nc]:
                yield nr, nc

    shortest_length = np.nan
    if passable[MAZE_START]:
        distances = {MAZE_START: 0}
        queue = deque([MAZE_START])
        while queue:
            cell = queue.popleft()
            for nxt in neighbors(cell):
                if nxt not in distances:
                    distances[nxt] = distances[cell] + 1
                    queue.append(nxt)
        if MAZE_GOAL in distances:
            shortest_length = float(distances[MAZE_GOAL])

    if not passable[MAZE_START]:
        return shortest_length, 0

    def heuristic(cell):
        return abs(cell[0] - MAZE_GOAL[0]) + abs(cell[1] - MAZE_GOAL[1])

    frontier = [(heuristic(MAZE_START), 0, MAZE_START)]
    best_cost = {MAZE_START: 0}
    closed = set()
    expansions = 0
    while frontier:
        _, cost, cell = heapq.heappop(frontier)
        if cell in closed or cost != best_cost.get(cell):
            continue
        closed.add(cell)
        expansions += 1
        if cell == MAZE_GOAL:
            break
        for nxt in neighbors(cell):
            new_cost = cost + 1
            if new_cost < best_cost.get(nxt, float("inf")):
                best_cost[nxt] = new_cost
                heapq.heappush(frontier, (new_cost + heuristic(nxt), new_cost, nxt))
    return shortest_length, expansions


maze_metrics_by_model = {}
for model_name, result in solvability_results.items():
    per_maze = [
        shortest_path_length_and_astar_expansions(maze) for maze in result["mazes"]
    ]
    lengths = np.asarray([item[0] for item in per_maze], dtype=float)
    expansions = np.asarray([item[1] for item in per_maze], dtype=float)
    maze_metrics_by_model[model_name] = {
        "shortest_solution_length": (
            float(np.nanmean(lengths)) if np.isfinite(lengths).any() else np.nan
        ),
        "astar_expansions": float(expansions.mean()),
        "solved_samples": int(np.isfinite(lengths).sum()),
    }


def plot_metric_boxplot(metric, title, y_title):
    fig = go.Figure()
    for objective_index, objective in enumerate(OBJECTIVES):
        names = [
            name
            for name in models
            if MODEL_META[name]["objective"] == objective
            and np.isfinite(maze_metrics_by_model[name][metric])
        ]
        values = np.asarray([maze_metrics_by_model[name][metric] for name in names])
        if not len(values):
            continue
        color = OBJECTIVE_COLORS[objective]
        fig.add_trace(
            go.Box(
                x=[objective_index] * len(values),
                y=values,
                name=OBJECTIVE_LABELS[objective],
                boxpoints="all",
                jitter=0.2,
                pointpos=0,
                marker=dict(color=color, size=8, opacity=0.78),
                line=dict(color=color, width=2),
                customdata=np.asarray(names)[:, None],
                hovertemplate="%{customdata[0]}<br>Model mean: %{y:.2f}<extra></extra>",
                showlegend=False,
            )
        )
        mean_value = float(values.mean())
        fig.add_shape(
            type="line",
            x0=objective_index - 0.18,
            x1=objective_index + 0.18,
            y0=mean_value,
            y1=mean_value,
            xref="x",
            yref="y",
            line=dict(color="#111827", width=4),
        )
        fig.add_annotation(
            x=objective_index,
            y=mean_value,
            text=(
                f"mean {mean_value:.2f}"
                if metric == "shortest_solution_length"
                else f"mean {mean_value:.1f}"
            ),
            showarrow=False,
            yshift=15,
            font=dict(size=13, color="#111827"),
            bgcolor="rgba(255,255,255,0.88)",
        )
    fig.update_layout(
        title=f"{title} ? per-model means over {N_METRIC_SAMPLES} mazes",
        xaxis=dict(
            title="Training objective",
            tickmode="array",
            tickvals=[0, 1],
            ticktext=[OBJECTIVE_LABELS[name] for name in OBJECTIVES],
            range=[-0.5, 1.5],
        ),
        yaxis=dict(title=y_title, rangemode="tozero"),
        template="plotly_white",
        width=1000,
        height=500,
    )
    fig.show()


plot_metric_boxplot(
    "shortest_solution_length", "Shortest solution length", "Grid moves"
)
plot_metric_boxplot("astar_expansions", "A* search effort until exit", "Expanded cells")

## 8. Maze-space accessibility

For every generated maze, measure what fraction of its white (passage) cells are reachable from the entrance, and count white connected components that do not include the entrance. Each box plot shows the per-model mean across generated samples, grouped by training objective.


In [33]:
OPTIMIZATION_START = (0, 1)
OPTIMIZATION_DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))

# If the sampling cell predates storing mazes, regenerate only the missing samples.
if "evaluation_initial_states" not in globals():
    evaluation_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 501)
    evaluation_initial_states = torch.randn(
        N_METRIC_SAMPLES,
        EMBEDDING_DIM,
        32,
        32,
        generator=evaluation_rng,
        device=DEVICE,
        dtype=binary_embeddings.dtype,
    )
for model_name, candidate in models.items():
    result = solvability_results.get(model_name)
    if result is None or "mazes" not in result:
        continuous = sample_flow(candidate, evaluation_initial_states, steps=ODE_STEPS)
        mazes = (
            torch.einsum("bdhw,kd->bkhw", continuous, binary_embeddings)
            .argmax(dim=1)
            .cpu()
            .numpy()
            .astype(np.uint8)
        )
        solved = np.asarray([solvability_rate([maze]) == 1.0 for maze in mazes])
        solvability_results[model_name] = {
            "objective": MODEL_META[model_name]["objective"],
            "replicate": MODEL_META[model_name]["replicate"],
            "solvable": solved,
            "mazes": mazes,
            "rate": float(solved.mean()),
        }


def maze_accessibility_metrics(maze):
    white = np.asarray(maze) == 1
    height, width = white.shape

    def neighbors(cell):
        row, col = cell
        for dr, dc in OPTIMIZATION_DIRECTIONS:
            nr, nc = row + dr, col + dc
            if 0 <= nr < height and 0 <= nc < width and white[nr, nc]:
                yield nr, nc

    white_cells = {(int(r), int(c)) for r, c in np.argwhere(white)}
    reachable = set()
    if OPTIMIZATION_START in white_cells:
        reachable.add(OPTIMIZATION_START)
        queue = [OPTIMIZATION_START]
        while queue:
            cell = queue.pop()
            for nxt in neighbors(cell):
                if nxt not in reachable:
                    reachable.add(nxt)
                    queue.append(nxt)

    # Count every white component outside the entrance component.
    unseen = set(white_cells)
    isolated_components = 0
    while unseen:
        seed = unseen.pop()
        component = {seed}
        queue = [seed]
        while queue:
            cell = queue.pop()
            for nxt in neighbors(cell):
                if nxt in unseen:
                    unseen.remove(nxt)
                    component.add(nxt)
                    queue.append(nxt)
        if not reachable or not (component & reachable):
            isolated_components += 1

    reachable_white_percent = (
        100.0 * len(reachable) / len(white_cells) if white_cells else 0.0
    )
    return reachable_white_percent, isolated_components


accessibility_by_model = {}
for model_name, result in solvability_results.items():
    per_maze = [maze_accessibility_metrics(maze) for maze in result["mazes"]]
    accessibility_by_model[model_name] = {
        "reachable_white_percent": float(np.mean([item[0] for item in per_maze])),
        "isolated_components": float(np.mean([item[1] for item in per_maze])),
    }


def plot_accessibility_boxplot(metric, title, y_title, y_range=None):
    fig = go.Figure()
    for objective_index, objective in enumerate(OBJECTIVES):
        names = [name for name in models if MODEL_META[name]["objective"] == objective]
        values = np.asarray([accessibility_by_model[name][metric] for name in names])
        color = OBJECTIVE_COLORS[objective]
        fig.add_trace(
            go.Box(
                x=[objective_index] * len(values),
                y=values,
                name=OBJECTIVE_LABELS[objective],
                boxpoints="all",
                jitter=0.2,
                pointpos=0,
                marker=dict(color=color, size=8, opacity=0.78),
                line=dict(color=color, width=2),
                customdata=np.asarray(names)[:, None],
                hovertemplate="%{customdata[0]}<br>Model mean: %{y:.2f}<extra></extra>",
                showlegend=False,
            )
        )
        mean_value = float(values.mean())
        fig.add_shape(
            type="line",
            x0=objective_index - 0.18,
            x1=objective_index + 0.18,
            y0=mean_value,
            y1=mean_value,
            xref="x",
            yref="y",
            line=dict(color="#111827", width=4),
        )
        fig.add_annotation(
            x=objective_index,
            y=mean_value,
            text=f"mean {mean_value:.2f}",
            showarrow=False,
            yshift=15,
            font=dict(size=13, color="#111827"),
            bgcolor="rgba(255,255,255,0.88)",
        )
    fig.update_layout(
        title=f"{title} ? per-model means over {N_METRIC_SAMPLES} mazes",
        xaxis=dict(
            title="Training objective",
            tickmode="array",
            tickvals=[0, 1],
            ticktext=[OBJECTIVE_LABELS[name] for name in OBJECTIVES],
            range=[-0.5, 1.5],
        ),
        yaxis=dict(title=y_title, rangemode="tozero", range=y_range),
        template="plotly_white",
        width=1000,
        height=500,
    )
    fig.show()


plot_accessibility_boxplot(
    "reachable_white_percent",
    "White cells reachable from entrance",
    "Reachable white cells (%)",
    y_range=[0, 100],
)
plot_accessibility_boxplot(
    "isolated_components",
    "White components isolated from entrance",
    "Isolated component count",
)